https://drive.google.com/file/d/1KTBrzhhtX4LozrMBPIqOcxGqx_JYJ7NY/view?usp=sharing
Файл, где есть проверки, что всё работает.

In [1]:
!pip install clickhouse-driver

  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import requests as req
import pandas as pd
import json
from datetime import datetime, timedelta
from clickhouse_driver import Client

In [13]:
URL = '''https://api.exchangerate.host/timeframe?access_key=043dc9dad696914726d3064e9d917294&source=USD&start_date=2023-01-01&end_date=2023-01-01'''

In [17]:
CH_CLIENT = Client(
    host='158.160.116.58',  # IP-адрес сервера ClickHouse
    user='student',  # Имя пользователя для подключения
    password='dfqh89fhq8',  # Пароль для подключения
    database='sandbox'
)

In [18]:
def extract_data(url, file_name):
    """
    Выгружаем данные из url в файл с именем file_name
    """
    response = req.get(url)
    with open(file_name, 'w', encoding='utf-8') as file:
        file.write(response.text)
    

In [ ]:
def transform_data(s_file, csv_file, date):
    """
    Работаем с данными в формате JSON.
    Преобразуем в табличные. 
    Записываем в csv файл
    """
    text = ''
    with open(s_file, 'r', encoding='utf-8') as file:
        text = file.read()
    data = json.loads(text)
    #print(type(data))
    #print(json.dumps(data, ensure_ascii=False, indent=4))
    #print(data['quotes']["2023-01-01"])
    transformed_data = []
    for key, value in data['quotes']["2023-01-01"].items():
        transformed_data.append({
            'date':"2023-01-01",
            'currency_source':'USD',
            'currency':key[3:],
            'value':value
        })
    df = pd.DataFrame(transformed_data)
    df.to_csv(csv_file, sep=",", encoding ='utf-8', index=False)
    


In [45]:
def upload_to_clickhouse(csv_file, table_name, client):
    """
    Считывает CSV файл
    Создаёт таблицу в clickhouse
    Добавляет данные из файла в clickhouse
    """
    data_frame = pd.read_csv(csv_file)
    client.execute(f'CREATE TABLE IF NOT EXISTS {table_name} (date String, currency_source String, currency String, value Float64) Engine = Log')
    client.execute(f'INSERT INTO {table_name} VALUES', data_frame.to_dict('records'))
    
